In [ ]:
import pandas as pd
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.metrics.pairwise import cosine_similarity

# Load and preprocess tweets
text = pd.read_csv("LP IV Updated Datasets/CBOW/twitter_dataset.csv").Text.dropna()
tweets = text.to_frame()
sentences = []
for tweet in text:
    words = simple_preprocess(tweet)
    sentences.append(words)
tweets["words"] = sentences
tweets = tweets[tweets.words.str.len() > 0]

# Train CBOW and average word vectors for each tweet
cbow = Word2Vec(tweets.words, min_count=1, sg=0, epochs=20)
vectors = []
for words in tweets.words:
    vector = cbow.wv.get_mean_vector(words)
    vectors.append(vector)

# Recommend the five closest tweets
words = simple_preprocess(input("Enter keywords: "))
query = cbow.wv.get_mean_vector(words)
tweets["score"] = cosine_similarity([query], vectors)[0]
print(tweets.nlargest(5, "score")[["Text", "score"]])
